# ECE 491 — Python Lab 5
## CIFAR Image Recognition Using a Shallow Convolutional Neural Network
### Complete professor-aligned solutions | All exercises | Detailed steps, formulas, executable code and PDF report

**Source:** The professor's five-page *Python Lab 5 — ECE 491* handout. The notebook answers the exercises **in their original order**, preserving the prescribed architecture, training procedure and augmentation requirements.

**How to use:** Open in Google Colab, choose **Runtime → Change runtime type → T4 GPU** if available, then **Runtime → Run all**. Download the PDF produced by the last cell only after the real TensorFlow training has finished.

**Do not fabricate numerical outputs.** The CIFAR-10 dataset has to be loaded and both CNNs must be trained before Q(training accuracy), Q(augmentation), and Q(first 512) have numerical answers. This notebook computes those results automatically.

### Conventions and important corrections to the handout

- **CIFAR-10:** 60,000 images **in total**, with **50,000 training** and **10,000 test**. The paragraph on page 2 saying `(60000,32,32,3)` for training is a typo.
- **Dataset sizes:** `x_train.shape=(50000,32,32,3)` and `y_train.shape=(50000,1)`; test shapes `(10000,32,32,3)` and `(10000,1)`.
- **Validation:** The handout explicitly uses `test_ds` for validation. We implement it as written, but this means the test set is *not an independent final evaluation sample*. In other projects, create an independent validation split.
- **Logits:** With `SparseCategoricalCrossentropy(from_logits=True)`, the last layer must output 10 **raw logits** (no Softmax). `argmax` gives the predicted class; Softmax is needed to convert logits into probabilities.
- **5 epochs / batch 256:** Exactly follow the supplied `steps_per_epoch=50000//256=195` and `validation_steps=10000//256=39`. The per-epoch validation uses 9,984 test images (39 batches); full `evaluate(test_ds)` uses all 10,000.
- **Augmentation ≈4%:** The handout says the accuracy increases by about 4%; that is a *reference outcome*, not a mathematical guarantee. We calculate and report the actual difference without inventing results.

Mathematics is displayed using `$$ ... $$` throughout.

# Part 0 — Setup, reproducibility and CIFAR-10 classes

The ten mutually exclusive CIFAR-10 categories are, in order: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. Each 32×32 colour image contains 3,072 scalar pixel values:

$$
32\times32\times3 = 3072.
$$

A GPU is optional for correctness, but helps reduce CNN training time. The following cell reports whether a GPU is visible; it does not claim one exists when none is available.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 491
BATCH_SIZE = 256
EPOCHS = 5
STEPS_PER_EPOCH = 50000 // BATCH_SIZE
VALIDATION_STEPS = 10000 // BATCH_SIZE
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

tf.keras.utils.set_random_seed(SEED)
print("Python version:", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))
print("Batch size:", BATCH_SIZE)
print("Training steps per epoch:", STEPS_PER_EPOCH)
print("Validation steps:", VALIDATION_STEPS)

# Question A — Load CIFAR-10 and print all NumPy array shapes

## Step 1: Load the professor's dataset

The handout specifies:

```python
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
```

## Step 2: Interpret the shapes

| Variable | Expected shape | Meaning |
|---|---|---|
| `x_train` | `(50000, 32, 32, 3)` | 50,000 colour images |
| `y_train` | `(50000, 1)` | One integer label per training image |
| `x_test` | `(10000, 32, 32, 3)` | 10,000 colour images |
| `y_test` | `(10000, 1)` | One integer label per test image |

A value in `y_train` is an integer from 0 to 9. It is **not** a one-hot vector.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Q A — x_train:", x_train.shape, x_train.dtype)
print("Q A — y_train:", y_train.shape, y_train.dtype)
print("Q A — x_test: ", x_test.shape, x_test.dtype)
print("Q A — y_test: ", y_test.shape, y_test.dtype)

assert x_train.shape == (50000, 32, 32, 3)
assert y_train.shape == (50000, 1)
assert x_test.shape == (10000, 32, 32, 3)
assert y_test.shape == (10000, 1)
assert set(np.unique(y_train)).issubset(set(range(10)))

# Question B — Display the first nine training images in a 3×3 subplot

## Mini-steps

1. Create a 3-row, 3-column grid with `plt.subplots(3, 3)`.
2. For each index `i=0,...,8`, display `x_train[i]`.
3. Use the corresponding CIFAR-10 class name `CLASS_NAMES[int(y_train[i,0])]`.
4. Hide coordinate ticks because the goal is to recognise the image category.

Every subplot shows an **original 32×32 RGB image**, rather than an artificial replacement image.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8.0, 8.0))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(f"Sample {i+1}: {CLASS_NAMES[int(y_train[i, 0])]}", fontsize=10)
    ax.axis("off")
fig.suptitle("ECE 491 Lab 5 — First Nine CIFAR-10 Training Images", fontsize=13)
fig.tight_layout()
fig.savefig("Lab5_First_Nine_Images.png", dpi=160, bbox_inches="tight")
plt.show()

# Question — Convert the NumPy arrays to a zipped `tf.data` dataset

The professor supplies three operations:

```python
x_train_ds = tf.data.Dataset.from_tensor_slices(x_train)
y_train_ds = tf.data.Dataset.from_tensor_slices(y_train)
train_ds = tf.data.Dataset.zip((x_train_ds, y_train_ds))
```

They form a dataset of individual `(image, label)` pairs. The 50,000 images do not change size. We flatten labels from shape `(N,1)` to `(N,)` for a clean sparse-classification target; the *label values stay exactly the same*.

**Analogy:** Each dataset entry is a sealed envelope containing one picture plus its correct class label. `zip` ensures the label stays with the correct picture.

In [ ]:
# Flatten only the label axis, not the images.
y_train_flat = y_train.reshape(-1)
y_test_flat = y_test.reshape(-1)

x_train_ds = tf.data.Dataset.from_tensor_slices(x_train)
y_train_ds = tf.data.Dataset.from_tensor_slices(y_train_flat)
raw_train_ds = tf.data.Dataset.zip((x_train_ds, y_train_ds))

first_image, first_label = next(iter(raw_train_ds))
print("Unbatched image shape:", first_image.shape)
print("Unbatched label shape:", first_label.shape)
print("First label:", int(first_label.numpy()))
assert tuple(first_image.shape) == (32, 32, 3)
assert int(first_label.numpy()) == int(y_train[0, 0])

# Question — Standardise every image with `Dataset.map`

The professor requires **casting to `float32`**, followed by:

```python
tf.image.per_image_standardization(image)
```

For a single image with pixel mean $\mu$ and standard deviation $\sigma$, TensorFlow's operation is conceptually

$$
\boxed{z_{p}=\frac{x_p-\mu}{\max(\sigma,1/\sqrt{3072})}}.
$$

The lower bound in the denominator makes standardisation numerically safe for nearly constant images. It normalises **each image separately**, not each channel across the entire dataset.

**Mini-steps:** convert → standardise → return the unchanged label. A normal image should have approximately zero mean and unit standard deviation after preprocessing.

In [ ]:
def preprocess_fn(image, label):
    image = tf.cast(image, dtype=tf.float32)
    image = tf.image.per_image_standardization(image)
    return image, label

standard_train_ds = raw_train_ds.map(preprocess_fn, num_parallel_calls=AUTOTUNE)
example_image, example_label = next(iter(standard_train_ds))
print("Standardised sample mean:", float(tf.reduce_mean(example_image)))
print("Standardised sample std:", float(tf.math.reduce_std(example_image)))
print("Label still unchanged:", int(example_label.numpy()))
assert example_image.dtype == tf.float32
assert abs(float(tf.reduce_mean(example_image))) < 2e-4

# Question — Shuffle, batch and repeat the training dataset

The professor's order is important:

$$
\text{map} \rightarrow \text{shuffle}(1000) \rightarrow \text{batch}(256) \rightarrow \text{repeat}.
$$

- `shuffle(1000)` uses a buffer of up to 1,000 examples; this is *not necessarily* a perfectly uniform shuffle of all 50,000 images.
- `batch(256)` produces mini-batches for training.
- `repeat()` makes the dataset infinite, so `model.fit` must have `steps_per_epoch` to know when to end each epoch.
- `prefetch(AUTOTUNE)` is a **performance-only** addition; it does not change the intended training set.

Professor's configuration:

$$
\text{steps per epoch}=\left\lfloor\frac{50000}{256}\right\rfloor=195.
$$

Because the batch operation uses its default `drop_remainder=False`, the final partial batch can be carried into the next epoch in the repeated stream. This is consistent with the provided code.

In [ ]:
train_ds = (
    standard_train_ds
    .shuffle(1000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(AUTOTUNE)
)

train_images_batch, train_labels_batch = next(iter(train_ds))
print("Train batch shape:", train_images_batch.shape)
print("Train labels shape:", train_labels_batch.shape)
assert train_images_batch.shape == (256, 32, 32, 3)
assert train_labels_batch.shape == (256,)

# Question — Create `test_ds` without shuffle or repeat

The handout explicitly says: **DO NOT shuffle or repeat the test data.**

We build the test dataset using precisely the same deterministic preprocessing function as the baseline training data:

$$
\text{test NumPy arrays} \rightarrow \text{zip} \rightarrow \text{standardise} \rightarrow \text{batch}.
$$

**Why keep the original order?** Later, the first 512 predictions must match the first 512 labels without reordering.

We keep `test_ds` unchanged for all experiments, including augmentation. The professor uses it during `model.fit(..., validation_data=test_ds)`.

In [ ]:
x_test_ds = tf.data.Dataset.from_tensor_slices(x_test)
y_test_ds = tf.data.Dataset.from_tensor_slices(y_test_flat)
raw_test_ds = tf.data.Dataset.zip((x_test_ds, y_test_ds))

test_ds = (
    raw_test_ds
    .map(preprocess_fn, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_batch_images, test_batch_labels = next(iter(test_ds))
print("Test batch shape:", test_batch_images.shape)
print("Test batch labels:", test_batch_labels.shape)
assert test_batch_images.shape == (BATCH_SIZE, 32, 32, 3)
assert np.array_equal(test_batch_labels.numpy(), y_test_flat[:BATCH_SIZE])
print("Test ds has no shuffle and no repeat.")

# Question — Construct the professor's exact shallow CNN architecture

The handout specifies the following layers **in this order** (no extra convolution, Dropout or Softmax activation):

| # | Layer | Parameters | Output shape |
|---|---|---|---|
| Input | RGB image | — | $32\times32\times3$ |
| 1 | Conv2D | 32 filters, 5×5, ReLU, valid padding | $28\times28\times32$ |
| 2 | MaxPooling2D | 2×2, stride 2 | $14\times14\times32$ |
| 3 | BatchNormalization | default settings | $14\times14\times32$ |
| 4 | Conv2D | 32 filters, 5×5, ReLU | $10\times10\times32$ |
| 5 | Conv2D | 32 filters, 5×5, ReLU | $6\times6\times32$ |
| 6 | MaxPooling2D | 2×2, stride 2 | $3\times3\times32$ |
| 7 | BatchNormalization | default settings | $3\times3\times32$ |
| 8 | Flatten | $3\cdot3\cdot32$ | 288 |
| 9 | Dense | 256 units, ReLU | 256 |
| 10 | Dense | 10 units, **linear logits** | 10 |

## Exact parameter calculation

For a convolution with $C_{\mathrm{in}}$ input channels, $C_{\mathrm{out}}$ filters and a 5×5 kernel:

$$
P_{\mathrm{conv}}=(5\cdot5\cdot C_{\mathrm{in}}+1)C_{\mathrm{out}}.
$$

The three convolution layers have **2,432**, **25,632**, and **25,632** parameters. Each BatchNorm layer has 128 total parameters (64 trainable and 64 nontrainable). The 256-unit Dense layer has $288\cdot256+256=73{,}984$ parameters, and the output Dense layer has $256\cdot10+10=2{,}570$.

$$
\boxed{\text{Total}=130{,}506;\quad \text{Trainable}=130{,}378;\quad\text{Nontrainable}=128.}
$$

These numbers use standard Keras BatchNorm settings and its initial moving-statistics variables.

In [ ]:
def build_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),
        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(10),  # 10 logits: NO Softmax here
    ], name="ECE491_Lab5_CNN")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["sparse_categorical_accuracy"],
    )
    return model

model = build_cnn_model()
model.summary()
print("Total parameters:", model.count_params())
print("Output shape:", model.output_shape)
assert model.count_params() == 130506
assert model.output_shape == (None, 10)
assert sum(isinstance(layer, tf.keras.layers.Conv2D) for layer in model.layers) == 3
assert sum(isinstance(layer, tf.keras.layers.BatchNormalization) for layer in model.layers) == 2

# Save the model's *initial* weights for a controlled augmentation comparison.
INITIAL_WEIGHTS = [weight.copy() for weight in model.get_weights()]

# Question — Compile and train the baseline CNN for five epochs

The professor requires:

```python
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss_fn,
              metrics=['sparse_categorical_accuracy'])
model.fit(train_ds, validation_data=test_ds,
          steps_per_epoch=50000//256, epochs=5,
          validation_steps=10000//256)
```

We use the **same computation**, saving the history and measuring full-test accuracy separately.

## Why sparse categorical cross-entropy?

With ten mutually exclusive classes and integer label $y_i\in\{0,\ldots,9\}$, the loss is

$$
\mathcal{L}= -\frac1N\sum_{i=1}^{N}\log\left(\frac{e^{z_{i,y_i}}}{\sum_{c=0}^{9}e^{z_{i,c}}}\right),
$$

where $z_{i,c}$ are the *raw logits*. `from_logits=True` uses a stable implementation of Softmax + cross-entropy internally.

**Report:** The numerical validation accuracy is an **output of the experiment**, not a fixed answer. The professor's stated settings do not uniquely determine an accuracy across GPU versions and random seeds.

In [ ]:
BASELINE_START = time.perf_counter()
history_base = model.fit(
    train_ds,
    validation_data=test_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_steps=VALIDATION_STEPS,
    verbose=2,
)
baseline_training_seconds = time.perf_counter() - BASELINE_START

base_loss, base_accuracy = model.evaluate(test_ds, verbose=0)  # ALL 10,000 images
print("Baseline last-epoch validation accuracy (%):",
      round(float(history_base.history["val_sparse_categorical_accuracy"][-1]) * 100, 2))
print("Baseline full 10,000-image test accuracy (%):", round(100*base_accuracy, 2))
print("Baseline full-test loss:", float(base_loss))
print("Baseline training seconds:", round(baseline_training_seconds, 2))

# Question — Visualise the baseline learning curves

Training loss and validation loss show whether the network fits the training data and whether it generalises to the professor's validation data. If the training curve improves but the validation loss worsens, this may indicate overfitting.

This is a supporting plot; it does not change the required five-epoch training algorithm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in [
    (axes[0], "loss", "Cross-entropy loss"),
    (axes[1], "sparse_categorical_accuracy", "Classification accuracy"),
]:
    ax.plot(history_base.history[metric], marker="o", label="Training")
    ax.plot(history_base.history["val_" + metric], marker="s", label="Validation")
    ax.set_xlabel("Epoch")
    ax.set_title("Baseline CNN — " + title)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig("Lab5_Baseline_Learning_Curves.png", dpi=170, bbox_inches="tight")
plt.show()

# Question — Implement data augmentation in the *training* pipeline

The professor requests exactly two augmentation techniques:

1. `tf.image.random_flip_left_right(image)` — randomly reverses an image horizontally. It is label-preserving for typical CIFAR-10 categories in this context.
2. `tf.image.random_contrast(image, lower=0.8, upper=1.2)` — randomly scales image contrast by a factor between 0.8 and 1.2.

**Source-aligned order:** Convert image to `float32` → per-image standardisation → random flip → random contrast → return the same class label.

**Important:** `test_ds` must remain *unchanged and deterministic*. Augmentation is **training-only**.

**Analogy:** Students learn more robustly from seeing the same object under different lighting and left/right orientations. The network should learn the object category, not memorise a particular pixel arrangement.

In [ ]:
def preprocess_new_fn(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.per_image_standardization(image)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, label

augmented_train_ds = (
    raw_train_ds
    .map(preprocess_new_fn, num_parallel_calls=AUTOTUNE)
    .shuffle(1000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(AUTOTUNE)
)

aug_batch, aug_labels = next(iter(augmented_train_ds))
print("Augmented training batch:", aug_batch.shape)
print("Unmodified test dataset batch:", next(iter(test_ds))[0].shape)
assert aug_batch.shape == (BATCH_SIZE, 32, 32, 3)
# No transformations are added to test_ds.

# Question — Retrain the same CNN with augmentation for five epochs

We build the **same** architecture with the **same** loss, Adam optimizer, batch size, steps and epochs. We copy the exact initial model weights to reduce the unfair advantage of different initialisation.

The professor's example states that augmentation increases accuracy by about 4%; this is **not guaranteed** for a particular run. We report both the percentage-point change and the relative percentage change:

$$
\Delta_{pp}=100\left(A_{aug}-A_{base}\right),
$$

$$
\Delta_{relative}=100\frac{A_{aug}-A_{base}}{A_{base}},
$$

where each $A$ is the fraction of correctly classified images on the **same full 10,000-image test set**.

In [ ]:
aug_model = build_cnn_model()
aug_model.set_weights(INITIAL_WEIGHTS)  # Same initial model weights

AUG_START = time.perf_counter()
history_aug = aug_model.fit(
    augmented_train_ds,
    validation_data=test_ds,      # exactly the same unmodified pipeline
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_steps=VALIDATION_STEPS,
    verbose=2,
)
augmented_training_seconds = time.perf_counter() - AUG_START

aug_loss, aug_accuracy = aug_model.evaluate(test_ds, verbose=0)
delta_pp = 100.0 * (aug_accuracy - base_accuracy)
delta_relative_percent = 100.0 * (aug_accuracy-base_accuracy)/base_accuracy if base_accuracy else float("nan")

print(f"Baseline full-test accuracy:   {100*base_accuracy:.2f}%")
print(f"Augmented full-test accuracy:  {100*aug_accuracy:.2f}%")
print(f"Accuracy change:              {delta_pp:+.2f} percentage points")
print(f"Relative change:              {delta_relative_percent:+.2f}%")
print(f"Augmented full-test loss:      {aug_loss:.5f}")
print(f"Augmented training seconds:   {augmented_training_seconds:.2f}")
print("Did augmentation improve full-test accuracy?", "Yes" if delta_pp > 0 else "No")

# Question — Compare both models and their five-epoch learning curves

Two independent sets of real training and validation curves are plotted. An augmentation improvement can stem from better exposure to varied training examples, but five epochs is short and the result varies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in [
    (axes[0], "loss", "Validation loss"),
    (axes[1], "sparse_categorical_accuracy", "Validation accuracy"),
]:
    ax.plot(history_base.history["val_" + metric], marker="o", label="No augmentation")
    ax.plot(history_aug.history["val_" + metric], marker="s", label="With augmentation")
    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig("Lab5_Baseline_Versus_Augmented.png", dpi=170, bbox_inches="tight")
plt.show()

# Final question — Predict the **first 512** test images

This question refers to the **retrained augmented model** after the augmentation experiment.

## Mini-steps

1. `test_ds` contains ordered, non-repeated batches of 256 original test images.
2. `test_ds.take(2)` yields precisely **512 images**.
3. `aug_model.predict(test_ds.take(2))` returns a matrix of **512 rows and 10 logits per row**.
4. `np.argmax(logits,axis=1)` selects the predicted class index.
5. Compare against `y_test_flat[:512]`, preserving the original dataset order.
6. Calculate the fraction correct and a 10×10 confusion matrix. Rows = true classes; columns = predicted classes.

$$
\boxed{\mathrm{Accuracy}_{512} = \frac{\#\text{ correct predictions}}{512}\times100\%.}
$$

The professor calls logits 'likelihoods'. Strictly, they are **un-normalised class scores**; probabilities can be computed with Softmax if desired. For `argmax`, Softmax is not needed.

In [ ]:
first_512_ds = test_ds.take(2)  # 2 complete batches of 256 = 512
first_512_logits = aug_model.predict(first_512_ds, verbose=0)
first_512_pred = np.argmax(first_512_logits, axis=1).astype(np.int64)
first_512_true = y_test_flat[:512].astype(np.int64)

assert first_512_logits.shape == (512, 10)
assert first_512_pred.shape == (512,)
assert first_512_true.shape == (512,)

correct_512 = int(np.count_nonzero(first_512_pred == first_512_true))
accuracy_512 = 100.0 * correct_512 / 512

confusion_512 = np.zeros((10, 10), dtype=np.int64)
np.add.at(confusion_512, (first_512_true, first_512_pred), 1)
assert confusion_512.shape == (10, 10)
assert int(confusion_512.sum()) == 512
assert int(np.trace(confusion_512)) == correct_512

print("Prediction logits shape:", first_512_logits.shape)
print("Correct out of 512:", correct_512)
print(f"Accuracy on first 512 samples: {accuracy_512:.2f}%")
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_512)

# Final question — Display the 10×10 confusion matrix with class labels

For class $i$ and predicted class $j$:

$$
C_{ij}=\#\{n:y_n=i\ \text{and}\ \hat y_n=j\}.
$$

The diagonal contains correctly classified images. All other cells are errors. The sum of every entry is **512**, not 10,000, because this question evaluates only the first 512 images.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
heat = ax.imshow(confusion_512, cmap="Blues")
fig.colorbar(heat, ax=ax, shrink=0.83, label="Number of images")
ax.set_xticks(range(10), CLASS_NAMES, rotation=48, ha="right")
ax.set_yticks(range(10), CLASS_NAMES)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(f"First 512 CIFAR-10 Test Images — Accuracy {accuracy_512:.2f}%")
for i in range(10):
    for j in range(10):
        value = int(confusion_512[i, j])
        if value:
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > confusion_512.max()*0.55 else "black",
                    fontsize=8)
fig.tight_layout()
fig.savefig("Lab5_First_512_Confusion_Matrix.png", dpi=170, bbox_inches="tight")
plt.show()

# Results summary — Populate only from executed TensorFlow results

The following table is generated from the actual models. It keeps three metrics distinct:

- last-epoch validation accuracy over **39 batches / 9,984 images**, as per the handout;
- full-test accuracy over **all 10,000 images**, for a consistent model comparison;
- the accuracy of the **first 512 images**, for the specific final question.

This makes the final report transparent and reproducible.

In [ ]:
from pprint import pprint

RESULTS = {
    "data_train_shape": tuple(x_train.shape),
    "data_train_labels_shape": tuple(y_train.shape),
    "data_test_shape": tuple(x_test.shape),
    "data_test_labels_shape": tuple(y_test.shape),
    "train_steps": STEPS_PER_EPOCH,
    "validation_steps": VALIDATION_STEPS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "parameters_total": int(model.count_params()),
    "baseline_validation_last_percent":
        100.0 * float(history_base.history["val_sparse_categorical_accuracy"][-1]),
    "baseline_full_test_percent": 100.0 * float(base_accuracy),
    "augmented_validation_last_percent":
        100.0 * float(history_aug.history["val_sparse_categorical_accuracy"][-1]),
    "augmented_full_test_percent": 100.0 * float(aug_accuracy),
    "absolute_change_percentage_points": float(delta_pp),
    "relative_change_percent": float(delta_relative_percent),
    "first_512_correct": int(correct_512),
    "first_512_accuracy_percent": float(accuracy_512),
    "first_512_confusion_matrix": confusion_512.tolist(),
    "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],
}

pprint(RESULTS)

# Professional answer sheet — What to include in the professor's submission

**A — Array shapes:** Report the four verified tuples. **B — Images:** Show the first-nine-image plot. **Test pipeline:** Explain that it is zipped, standardised, batched, and never shuffled/repeated. **Model:** Show the ten prescribed layers and confirm output shape `(None,10)` and 130,506 parameters. **Baseline:** Report the five-epoch validation accuracy and full-test accuracy. **Augmentation:** Explain the random flip and contrast operations, preserve the original test pipeline, report the new accuracy and the observed change. **First 512:** Provide `model.predict` logits shape, correctly classified count, exact percentage and labelled 10×10 confusion matrix.

The professor's appendix asks for code; the final PDF generator appends the complete runnable experiment code after the results. It does **not** include unmeasured or fabricated values.

## Final theory cheat sheet

| Concept | Core idea |
|---|---|
| RGB images | Each sample has shape 32×32×3 |
| CIFAR-10 | 50,000 train + 10,000 test, 10 classes |
| `tf.data.Dataset.zip` | Combine image and label without misalignment |
| `map` | Apply one preprocessing function to every example |
| Per-image standardisation | Centre and scale each image independently |
| Shuffle vs repeat | Shuffle randomises training order; repeat allows multi-epoch reading |
| Conv2D with 5×5 filters | Learn spatial patterns; 32 output feature maps |
| MaxPooling2D | Reduce spatial dimensions, 2×2 stride 2 |
| BatchNorm | Learn scale and shift, track moving statistics |
| Flatten | Convert 3×3×32 into 288 features |
| Dense | Fully connected feature transformation |
| Cross-entropy from logits | Suitable for 10 integer-labelled classes; no output Softmax required |
| Data augmentation | Flip + contrast only on training data |
| Full test vs 512 | Separate the 10,000-image metric from the 512-image question |
| Confusion matrix | True class as row; predicted class as column |

### Five things to check before submitting

1. The CNN has exactly **three convolutional layers** in the professor's specified order.
2. The two networks train for exactly **five epochs**, not ten.
3. The baseline and augmentation use **the same unmodified `test_ds`**.
4. The confusion matrix sums to **512** and its trace equals the number correct.
5. The report includes the **actual measured accuracies**, plots, and complete Python appendix.

In [ ]:
# Generate a single professor-style PDF with measured answers and code appendix.
# All values below were calculated in the preceding cells from real CIFAR-10 data.
import textwrap
from matplotlib.backends.backend_pdf import PdfPages

SUBMISSION_CODE = '# ECE 491 Python Lab 5 — Complete experiment\n# Run with Python 3 and TensorFlow installed. Requires CIFAR-10 download.\n\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport tensorflow as tf\n\nSEED = 491\nBATCH_SIZE = 256\nEPOCHS = 5\nSTEPS_PER_EPOCH = 50000 // BATCH_SIZE\nVALIDATION_STEPS = 10000 // BATCH_SIZE\nAUTOTUNE = tf.data.AUTOTUNE\nCLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",\n               "dog", "frog", "horse", "ship", "truck"]\n\ntf.keras.utils.set_random_seed(SEED)\nprint("Python version:", sys.version.split()[0])\nprint("TensorFlow version:", tf.__version__)\nprint("GPUs detected:", tf.config.list_physical_devices("GPU"))\nprint("Batch size:", BATCH_SIZE)\nprint("Training steps per epoch:", STEPS_PER_EPOCH)\nprint("Validation steps:", VALIDATION_STEPS)\n\n(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()\n\nprint("Q A — x_train:", x_train.shape, x_train.dtype)\nprint("Q A — y_train:", y_train.shape, y_train.dtype)\nprint("Q A — x_test: ", x_test.shape, x_test.dtype)\nprint("Q A — y_test: ", y_test.shape, y_test.dtype)\n\nassert x_train.shape == (50000, 32, 32, 3)\nassert y_train.shape == (50000, 1)\nassert x_test.shape == (10000, 32, 32, 3)\nassert y_test.shape == (10000, 1)\nassert set(np.unique(y_train)).issubset(set(range(10)))\n\nfig, axes = plt.subplots(3, 3, figsize=(8.0, 8.0))\nfor i, ax in enumerate(axes.flat):\n    ax.imshow(x_train[i])\n    ax.set_title(f"Sample {i+1}: {CLASS_NAMES[int(y_train[i, 0])]}", fontsize=10)\n    ax.axis("off")\nfig.suptitle("ECE 491 Lab 5 — First Nine CIFAR-10 Training Images", fontsize=13)\nfig.tight_layout()\nfig.savefig("Lab5_First_Nine_Images.png", dpi=160, bbox_inches="tight")\nplt.show()\n\n# Flatten only the label axis, not the images.\ny_train_flat = y_train.reshape(-1)\ny_test_flat = y_test.reshape(-1)\n\nx_train_ds = tf.data.Dataset.from_tensor_slices(x_train)\ny_train_ds = tf.data.Dataset.from_tensor_slices(y_train_flat)\nraw_train_ds = tf.data.Dataset.zip((x_train_ds, y_train_ds))\n\nfirst_image, first_label = next(iter(raw_train_ds))\nprint("Unbatched image shape:", first_image.shape)\nprint("Unbatched label shape:", first_label.shape)\nprint("First label:", int(first_label.numpy()))\nassert tuple(first_image.shape) == (32, 32, 3)\nassert int(first_label.numpy()) == int(y_train[0, 0])\n\ndef preprocess_fn(image, label):\n    image = tf.cast(image, dtype=tf.float32)\n    image = tf.image.per_image_standardization(image)\n    return image, label\n\nstandard_train_ds = raw_train_ds.map(preprocess_fn, num_parallel_calls=AUTOTUNE)\nexample_image, example_label = next(iter(standard_train_ds))\nprint("Standardised sample mean:", float(tf.reduce_mean(example_image)))\nprint("Standardised sample std:", float(tf.math.reduce_std(example_image)))\nprint("Label still unchanged:", int(example_label.numpy()))\nassert example_image.dtype == tf.float32\nassert abs(float(tf.reduce_mean(example_image))) < 2e-4\n\ntrain_ds = (\n    standard_train_ds\n    .shuffle(1000, seed=SEED, reshuffle_each_iteration=True)\n    .batch(BATCH_SIZE)\n    .repeat()\n    .prefetch(AUTOTUNE)\n)\n\ntrain_images_batch, train_labels_batch = next(iter(train_ds))\nprint("Train batch shape:", train_images_batch.shape)\nprint("Train labels shape:", train_labels_batch.shape)\nassert train_images_batch.shape == (256, 32, 32, 3)\nassert train_labels_batch.shape == (256,)\n\nx_test_ds = tf.data.Dataset.from_tensor_slices(x_test)\ny_test_ds = tf.data.Dataset.from_tensor_slices(y_test_flat)\nraw_test_ds = tf.data.Dataset.zip((x_test_ds, y_test_ds))\n\ntest_ds = (\n    raw_test_ds\n    .map(preprocess_fn, num_parallel_calls=AUTOTUNE)\n    .batch(BATCH_SIZE)\n    .prefetch(AUTOTUNE)\n)\n\ntest_batch_images, test_batch_labels = next(iter(test_ds))\nprint("Test batch shape:", test_batch_images.shape)\nprint("Test batch labels:", test_batch_labels.shape)\nassert test_batch_images.shape == (BATCH_SIZE, 32, 32, 3)\nassert np.array_equal(test_batch_labels.numpy(), y_test_flat[:BATCH_SIZE])\nprint("Test ds has no shuffle and no repeat.")\n\ndef build_cnn_model():\n    model = tf.keras.Sequential([\n        tf.keras.layers.Input(shape=(32, 32, 3)),\n        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),\n        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),\n        tf.keras.layers.BatchNormalization(),\n        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),\n        tf.keras.layers.Conv2D(32, (5, 5), padding="valid", activation="relu"),\n        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),\n        tf.keras.layers.BatchNormalization(),\n        tf.keras.layers.Flatten(),\n        tf.keras.layers.Dense(256, activation="relu"),\n        tf.keras.layers.Dense(10),  # 10 logits: NO Softmax here\n    ], name="ECE491_Lab5_CNN")\n    model.compile(\n        optimizer=tf.keras.optimizers.Adam(),\n        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),\n        metrics=["sparse_categorical_accuracy"],\n    )\n    return model\n\nmodel = build_cnn_model()\nmodel.summary()\nprint("Total parameters:", model.count_params())\nprint("Output shape:", model.output_shape)\nassert model.count_params() == 130506\nassert model.output_shape == (None, 10)\nassert sum(isinstance(layer, tf.keras.layers.Conv2D) for layer in model.layers) == 3\nassert sum(isinstance(layer, tf.keras.layers.BatchNormalization) for layer in model.layers) == 2\n\n# Save the model\'s *initial* weights for a controlled augmentation comparison.\nINITIAL_WEIGHTS = [weight.copy() for weight in model.get_weights()]\n\nBASELINE_START = time.perf_counter()\nhistory_base = model.fit(\n    train_ds,\n    validation_data=test_ds,\n    steps_per_epoch=STEPS_PER_EPOCH,\n    epochs=EPOCHS,\n    validation_steps=VALIDATION_STEPS,\n    verbose=2,\n)\nbaseline_training_seconds = time.perf_counter() - BASELINE_START\n\nbase_loss, base_accuracy = model.evaluate(test_ds, verbose=0)  # ALL 10,000 images\nprint("Baseline last-epoch validation accuracy (%):",\n      round(float(history_base.history["val_sparse_categorical_accuracy"][-1]) * 100, 2))\nprint("Baseline full 10,000-image test accuracy (%):", round(100*base_accuracy, 2))\nprint("Baseline full-test loss:", float(base_loss))\nprint("Baseline training seconds:", round(baseline_training_seconds, 2))\n\nfig, axes = plt.subplots(1, 2, figsize=(11, 4))\nfor ax, metric, title in [\n    (axes[0], "loss", "Cross-entropy loss"),\n    (axes[1], "sparse_categorical_accuracy", "Classification accuracy"),\n]:\n    ax.plot(history_base.history[metric], marker="o", label="Training")\n    ax.plot(history_base.history["val_" + metric], marker="s", label="Validation")\n    ax.set_xlabel("Epoch")\n    ax.set_title("Baseline CNN — " + title)\n    ax.grid(True, alpha=0.3)\n    ax.legend()\nfig.tight_layout()\nfig.savefig("Lab5_Baseline_Learning_Curves.png", dpi=170, bbox_inches="tight")\nplt.show()\n\ndef preprocess_new_fn(image, label):\n    image = tf.cast(image, tf.float32)\n    image = tf.image.per_image_standardization(image)\n    image = tf.image.random_flip_left_right(image)\n    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)\n    return image, label\n\naugmented_train_ds = (\n    raw_train_ds\n    .map(preprocess_new_fn, num_parallel_calls=AUTOTUNE)\n    .shuffle(1000, seed=SEED, reshuffle_each_iteration=True)\n    .batch(BATCH_SIZE)\n    .repeat()\n    .prefetch(AUTOTUNE)\n)\n\naug_batch, aug_labels = next(iter(augmented_train_ds))\nprint("Augmented training batch:", aug_batch.shape)\nprint("Unmodified test dataset batch:", next(iter(test_ds))[0].shape)\nassert aug_batch.shape == (BATCH_SIZE, 32, 32, 3)\n# No transformations are added to test_ds.\n\naug_model = build_cnn_model()\naug_model.set_weights(INITIAL_WEIGHTS)  # Same initial model weights\n\nAUG_START = time.perf_counter()\nhistory_aug = aug_model.fit(\n    augmented_train_ds,\n    validation_data=test_ds,      # exactly the same unmodified pipeline\n    steps_per_epoch=STEPS_PER_EPOCH,\n    epochs=EPOCHS,\n    validation_steps=VALIDATION_STEPS,\n    verbose=2,\n)\naugmented_training_seconds = time.perf_counter() - AUG_START\n\naug_loss, aug_accuracy = aug_model.evaluate(test_ds, verbose=0)\ndelta_pp = 100.0 * (aug_accuracy - base_accuracy)\ndelta_relative_percent = 100.0 * (aug_accuracy-base_accuracy)/base_accuracy if base_accuracy else float("nan")\n\nprint(f"Baseline full-test accuracy:   {100*base_accuracy:.2f}%")\nprint(f"Augmented full-test accuracy:  {100*aug_accuracy:.2f}%")\nprint(f"Accuracy change:              {delta_pp:+.2f} percentage points")\nprint(f"Relative change:              {delta_relative_percent:+.2f}%")\nprint(f"Augmented full-test loss:      {aug_loss:.5f}")\nprint(f"Augmented training seconds:   {augmented_training_seconds:.2f}")\nprint("Did augmentation improve full-test accuracy?", "Yes" if delta_pp > 0 else "No")\n\nfig, axes = plt.subplots(1, 2, figsize=(11, 4))\nfor ax, metric, title in [\n    (axes[0], "loss", "Validation loss"),\n    (axes[1], "sparse_categorical_accuracy", "Validation accuracy"),\n]:\n    ax.plot(history_base.history["val_" + metric], marker="o", label="No augmentation")\n    ax.plot(history_aug.history["val_" + metric], marker="s", label="With augmentation")\n    ax.set_xlabel("Epoch")\n    ax.set_title(title)\n    ax.grid(True, alpha=0.3)\n    ax.legend()\nfig.tight_layout()\nfig.savefig("Lab5_Baseline_Versus_Augmented.png", dpi=170, bbox_inches="tight")\nplt.show()\n\nfirst_512_ds = test_ds.take(2)  # 2 complete batches of 256 = 512\nfirst_512_logits = aug_model.predict(first_512_ds, verbose=0)\nfirst_512_pred = np.argmax(first_512_logits, axis=1).astype(np.int64)\nfirst_512_true = y_test_flat[:512].astype(np.int64)\n\nassert first_512_logits.shape == (512, 10)\nassert first_512_pred.shape == (512,)\nassert first_512_true.shape == (512,)\n\ncorrect_512 = int(np.count_nonzero(first_512_pred == first_512_true))\naccuracy_512 = 100.0 * correct_512 / 512\n\nconfusion_512 = np.zeros((10, 10), dtype=np.int64)\nnp.add.at(confusion_512, (first_512_true, first_512_pred), 1)\nassert confusion_512.shape == (10, 10)\nassert int(confusion_512.sum()) == 512\nassert int(np.trace(confusion_512)) == correct_512\n\nprint("Prediction logits shape:", first_512_logits.shape)\nprint("Correct out of 512:", correct_512)\nprint(f"Accuracy on first 512 samples: {accuracy_512:.2f}%")\nprint("Confusion matrix (rows=true, columns=predicted):")\nprint(confusion_512)\n\nfig, ax = plt.subplots(figsize=(10, 8))\nheat = ax.imshow(confusion_512, cmap="Blues")\nfig.colorbar(heat, ax=ax, shrink=0.83, label="Number of images")\nax.set_xticks(range(10), CLASS_NAMES, rotation=48, ha="right")\nax.set_yticks(range(10), CLASS_NAMES)\nax.set_xlabel("Predicted class")\nax.set_ylabel("True class")\nax.set_title(f"First 512 CIFAR-10 Test Images — Accuracy {accuracy_512:.2f}%")\nfor i in range(10):\n    for j in range(10):\n        value = int(confusion_512[i, j])\n        if value:\n            ax.text(j, i, str(value), ha="center", va="center",\n                    color="white" if value > confusion_512.max()*0.55 else "black",\n                    fontsize=8)\nfig.tight_layout()\nfig.savefig("Lab5_First_512_Confusion_Matrix.png", dpi=170, bbox_inches="tight")\nplt.show()\n\nfrom pprint import pprint\n\nRESULTS = {\n    "data_train_shape": tuple(x_train.shape),\n    "data_train_labels_shape": tuple(y_train.shape),\n    "data_test_shape": tuple(x_test.shape),\n    "data_test_labels_shape": tuple(y_test.shape),\n    "train_steps": STEPS_PER_EPOCH,\n    "validation_steps": VALIDATION_STEPS,\n    "epochs": EPOCHS,\n    "batch_size": BATCH_SIZE,\n    "parameters_total": int(model.count_params()),\n    "baseline_validation_last_percent":\n        100.0 * float(history_base.history["val_sparse_categorical_accuracy"][-1]),\n    "baseline_full_test_percent": 100.0 * float(base_accuracy),\n    "augmented_validation_last_percent":\n        100.0 * float(history_aug.history["val_sparse_categorical_accuracy"][-1]),\n    "augmented_full_test_percent": 100.0 * float(aug_accuracy),\n    "absolute_change_percentage_points": float(delta_pp),\n    "relative_change_percent": float(delta_relative_percent),\n    "first_512_correct": int(correct_512),\n    "first_512_accuracy_percent": float(accuracy_512),\n    "first_512_confusion_matrix": confusion_512.tolist(),\n    "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],\n}\n\npprint(RESULTS)\n'
source_py = Path('ECE491_Lab5_Complete_Experiment.py')
source_py.write_text(SUBMISSION_CODE, encoding='utf-8')

submission_pdf = Path('ECE491_Lab5_Final_Submission.pdf')

def report_page(pdf, header, paragraphs):
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor('white')
    fig.text(0.07, 0.955, 'ECE 491  |  Python Lab 5', fontsize=11, color='#2a5170', weight='bold')
    fig.text(0.07, 0.92, header, fontsize=15, weight='bold')
    y = 0.875
    for paragraph in paragraphs:
        lines = textwrap.wrap(paragraph, width=95, break_long_words=False,
                              break_on_hyphens=False) or ['']
        for line in lines:
            fig.text(0.077, y, line, fontsize=9.0, va='top')
            y -= 0.020
        y -= 0.018
    plt.axis('off')
    pdf.savefig(fig)
    plt.close(fig)

def figure_page(pdf, file_path, title):
    image = plt.imread(file_path)
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.07, 0.94, 'ECE 491  |  ' + title, fontsize=13, weight='bold')
    ax = fig.add_axes([0.08, 0.10, 0.84, 0.78])
    ax.imshow(image)
    ax.axis('off')
    pdf.savefig(fig)
    plt.close(fig)

with PdfPages(submission_pdf) as pdf:
    report_page(pdf, 'CIFAR-10: Dataset and preprocessing', [
        'Dataset: 50,000 training RGB images and 10,000 test RGB images; 10 classes.',
        f"Image/label shapes: train {x_train.shape} and {y_train.shape}; test {x_test.shape} and {y_test.shape}.",
        'Training pipeline: zipped image-label pairs, per-image standardisation, shuffle(1000), batch(256), repeat, prefetch.',
        'Test pipeline: zipped image-label pairs, identical deterministic standardisation, batch(256); no shuffle, repeat or augmentation.',
        'CNN architecture: Conv2D(32,5x5,ReLU), MaxPool(2x2), BatchNorm, Conv2D(32,5x5,ReLU), Conv2D(32,5x5,ReLU), MaxPool(2x2), BatchNorm, Flatten, Dense(256,ReLU), Dense(10,linear logits).',
        f'Trainable+nontrainable model parameters: {model.count_params():,}.',
        'Loss/optimiser: sparse categorical cross-entropy (from_logits=True); Adam; 5 epochs; batch=256; 195 training steps and 39 validation steps per epoch.',
        'Professor uses the test dataset for validation; this limits its independence as an unbiased generalisation test.'
    ])
    figure_page(pdf, 'Lab5_First_Nine_Images.png', 'First nine training samples')
    report_page(pdf, 'Training and augmentation results', [
        f"Baseline last-epoch validation accuracy (9,984 samples): {100*history_base.history['val_sparse_categorical_accuracy'][-1]:.2f}%.",
        f'Baseline full 10,000-image test accuracy: {100*base_accuracy:.2f}%; test loss: {base_loss:.5f}.',
        'Augmentation: random horizontal flipping and random contrast factors 0.8 to 1.2, applied only to training; test pipeline unchanged.',
        f"Augmented last-epoch validation accuracy (9,984 samples): {100*history_aug.history['val_sparse_categorical_accuracy'][-1]:.2f}%.",
        f'Augmented full 10,000-image test accuracy: {100*aug_accuracy:.2f}%; test loss: {aug_loss:.5f}.',
        f'Observed accuracy change: {delta_pp:+.2f} percentage points ({delta_relative_percent:+.2f}% relative).',
        f'Baseline training time: {baseline_training_seconds:.2f} seconds; augmented training time: {augmented_training_seconds:.2f} seconds.',
        f'First 512 test images: {correct_512}/512 correct = {accuracy_512:.2f}%; output logits have shape {first_512_logits.shape}.',
        'The 10x10 confusion matrix has true labels as rows and predicted labels as columns; its entries total 512.'
    ])
    figure_page(pdf, 'Lab5_Baseline_Learning_Curves.png', 'Baseline learning curves')
    figure_page(pdf, 'Lab5_Baseline_Versus_Augmented.png', 'Validation: Baseline vs Augmented')
    figure_page(pdf, 'Lab5_First_512_Confusion_Matrix.png', 'First 512: Confusion matrix')
    report_page(pdf, 'Exact numerical confusion matrix', [
        'Class order: ' + ', '.join(CLASS_NAMES),
        'Each row corresponds to a true class and each column to a predicted class.',
        *[f'{CLASS_NAMES[i]}: ' + ' '.join(str(int(v)) for v in confusion_512[i]) for i in range(10)]
    ])
    code_lines = SUBMISSION_CODE.splitlines()
    pages = [code_lines[i:i+58] for i in range(0, len(code_lines), 58)]
    for page_num, page_lines in enumerate(pages, 1):
        fig = plt.figure(figsize=(8.27, 11.69))
        fig.text(0.06, 0.963, f'Appendix — Executable Python source ({page_num}/{len(pages)})',
                 fontsize=11, weight='bold')
        for line_num, src_line in enumerate(page_lines):
            # Keep the code legible and avoid running past the right page margin.
            if len(src_line) > 112:
                src_line = src_line[:109] + '...'
            fig.text(0.06, 0.937 - line_num*0.0155, src_line,
                     family='monospace', fontsize=6.65, va='top')
        plt.axis('off')
        pdf.savefig(fig)
        plt.close(fig)

print('Created:', submission_pdf.resolve())
print('Created:', source_py.resolve())
print('No experimental results were estimated; all reported accuracies were computed above.')
try:
    from google.colab import files
    files.download(str(submission_pdf))
    files.download(str(source_py))
except ImportError:
    print('Open the Files panel to download the PDF and Python file.')